# 01 · Reconciliation with official DfT totals (RAS0101)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam843/uk-road-accident-analytics/blob/main/notebooks/01_reconciliation.ipynb)

This is the data-quality proof for the whole project. Yearly totals from the cleaned pipeline
output are compared with DfT table **RAS0101** (*Reported road collisions, casualties and
vehicles involved, Great Britain*).

What RAS0101 publishes (sheet → column):

| Measure | RAS0101 | Pipeline |
|---|---|---|
| All collisions | Collisions → All collisions | rows in `collisions_2005_2019.parquet` |
| Fatal collisions | Collisions → Fatal | `collision_severity == 1` |
| Fatal or serious collisions (unadjusted) | Collisions → FSC (unadjusted) | severity 1 or 2 |
| Fatal or serious collisions (adjusted) | Collisions → FSC (adjusted) | fatal + Σ `collision_adjusted_serious` |
| Serious collisions (unadjusted / adjusted) | FSC − Fatal (derived) | same derivation |
| People killed | Casualties → All road users (killed) | `casualty_severity == 1` |
| All casualties | Casualties → All road users (all severities) | rows in `casualties_2005_2019.parquet` |

RAS0101 does not publish serious **casualties** on their own, so serious injury is checked at
collision level.

**Acceptance rule:** yearly collision counts must match within **0.5%**. Results: **TBD
(run notebooks/01_reconciliation.ipynb)**.

In [ ]:
# Setup: works from the repo root, from notebooks/, or on Google Colab.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("src/config.py").exists():
    if not Path("uk-road-accident-analytics").exists():
        !git clone -q https://github.com/maryam843/uk-road-accident-analytics.git
    %cd uk-road-accident-analytics
    %pip install -q -r requirements.txt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd

from src import clean, config, evaluate

config.setup_logging(logging.INFO)
evaluate.apply_style()
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

DATA_DIR = config.get_data_dir()
RAW, PROCESSED = DATA_DIR / "raw", DATA_DIR / "processed"
if not (PROCESSED / config.COLLISIONS_PARQUET).exists():
    # Fresh runtime (e.g. Colab): download (~2.5 GB) and clean. Takes a while once.
    from src import download

    download.main([])
    clean.run()
print("DATA_DIR =", DATA_DIR)

## Cleaning log
Counts recorded by `python -m src.clean` (duplicates dropped, invalid speed limits, flagged coordinates, ...).

In [ ]:
import json

clean_log = json.loads((PROCESSED / config.CLEAN_LOG_FILE).read_text())
coll_log = {k: v for k, v in clean_log["collisions"].items() if k != "decode"}
print(json.dumps(coll_log, indent=2))
print(json.dumps({k: v for k, v in clean_log["casualties"].items() if k != "decode"}, indent=2))
decode = pd.DataFrame(clean_log["collisions"]["decode"]).T
display(decode.sort_values("missing_to_nan", ascending=False))

## Pipeline vs RAS0101

In [ ]:
yearly = pd.read_parquet(PROCESSED / config.YEARLY_PARQUET)
official = clean.load_ras0101(RAW / config.RAS0101_RAW_FILE).loc[config.YEAR_MIN:config.YEAR_MAX]

for frame in (yearly, official):
    frame["serious_collisions_derived"] = frame["ksi_collisions"] - frame["fatal_collisions"]
    frame["serious_collisions_adjusted_derived"] = (
        frame["ksi_collisions_adjusted"] - frame["fatal_collisions"]
    )

MEASURES = {
    "collisions": "All collisions",
    "fatal_collisions": "Fatal collisions",
    "ksi_collisions": "Fatal or serious collisions (unadjusted)",
    "ksi_collisions_adjusted": "Fatal or serious collisions (adjusted)",
    "serious_collisions_derived": "Serious collisions (unadjusted)",
    "serious_collisions_adjusted_derived": "Serious collisions (adjusted)",
    "killed": "People killed",
    "casualties": "All casualties",
}
rows = []
for col, label in MEASURES.items():
    for year in official.index:
        p, o = float(yearly.loc[year, col]), float(official.loc[year, col])
        rows.append({"measure": label, "year": year, "pipeline": p, "ras0101": o,
                     "difference": p - o, "pct_difference": 100 * (p - o) / o})
recon = pd.DataFrame(rows)
recon.to_csv(PROCESSED / "reconciliation_ras0101.csv", index=False)

In [ ]:
collision_table = recon[recon["measure"] == "All collisions"].set_index("year")
display(collision_table[["pipeline", "ras0101", "difference", "pct_difference"]]
        .style.format({"pipeline": "{:,.0f}", "ras0101": "{:,.0f}", "difference": "{:+,.0f}",
                       "pct_difference": "{:+.3f}%"}))

In [ ]:
summary = recon.groupby("measure", sort=False).agg(
    max_abs_pct_difference=("pct_difference", lambda s: s.abs().max()),
    total_pipeline=("pipeline", "sum"),
    total_ras0101=("ras0101", "sum"),
)
summary["total_pct_difference"] = 100 * (summary["total_pipeline"] / summary["total_ras0101"] - 1)
display(summary.style.format("{:,.3f}"))

In [ ]:
# Headline check: collision counts within 0.5% every year.
worst = collision_table["pct_difference"].abs().max()
print(f"Largest yearly difference in collision counts: {worst:.3f}%")
assert worst <= 0.5, f"collision counts differ from RAS0101 by up to {worst:.3f}% (> 0.5%)"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.plot(collision_table.index, collision_table["ras0101"] / 1000, label="RAS0101 (official)")
ax.plot(collision_table.index, collision_table["pipeline"] / 1000, linestyle="--", marker="o",
        markersize=4, label="This pipeline")
ax.set_title("Reported road collisions per year, GB")
ax.set_ylabel("Collisions (thousands)")
ax.set_xlabel("Year")
ax.legend()

ax = axes[1]
pivot = recon.pivot(index="year", columns="measure", values="pct_difference")
for label in ["All collisions", "Fatal collisions", "People killed", "All casualties"]:
    ax.plot(pivot.index, pivot[label], marker="o", markersize=4, label=label)
ax.axhline(0, color="#52514e", linewidth=0.8)
ax.axhspan(-0.5, 0.5, color="#d9d9d6", alpha=0.5, label="±0.5% tolerance")
ax.set_title("Pipeline minus RAS0101")
ax.set_ylabel("Difference (% of official)")
ax.set_xlabel("Year")
ax.legend(fontsize=8)
fig.tight_layout()
evaluate.save_fig(fig, "01_reconciliation_collisions")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(yearly.index, yearly["ksi_collisions"] / 1000, label="Unadjusted (as reported)")
ax.plot(yearly.index, yearly["ksi_collisions_adjusted"] / 1000,
        label="Adjusted for changes in police severity reporting (DfT)")
ax.set_title("Fatal or serious collisions per year, GB")
ax.set_ylabel("Collisions (thousands)")
ax.set_xlabel("Year")
ax.legend()
fig.text(0.01, -0.03, f"Adjusted series: {config.SEVERITY_ADJUSTED_FOOTNOTE}. {config.ATTRIBUTION}",
         fontsize=7, color="#52514e", wrap=True)
evaluate.save_fig(fig, "01_ksi_collisions_adjusted_vs_unadjusted")
plt.show()

## Explaining any gaps

The tables above print the size of every difference. Possible reasons for a non-zero gap:

- **DfT revisions.** The open-data CSV is the latest revised version of every year. The
  data guide lists record-level changes (`historical_revisions`). A RAS table published at a
  different time can differ slightly. Use RAS0101 from the same release as the CSV
  (`DOWNLOAD_LOG.json` records both download dates).
- **De-duplication.** Rows dropped as duplicate collision indexes are listed in the cleaning
  log above. If the count is non-zero and the pipeline is *below* RAS0101 by the same amount,
  the duplicates were real records with a reused index, not copies.
- **Casualty linkage.** Casualties are kept only if their collision survived cleaning, so a
  casualty gap can come from collision-level drops.
- **Adjusted serious.** The adjusted figures are sums of probabilities, so they are not
  integers. Small differences can come from rounding in the published table.

Interpretation of this run: **TBD (run notebooks/01_reconciliation.ipynb)**.